# Target Hardware Benchmarking Notebook (Kaggle / Colab)
Measures seconds per 1,000 frames on target GPU hardware (e.g., Nvidia T4 / P100),
measures peak GPU memory and process RAM, and exports `timing_results.json`.

In [ ]:
# 1. Environment and Hardware Inspection
import os, sys, time, json, pathlib, subprocess, psutil, torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Hardware: {gpu_name}')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')

In [ ]:
# 2. Benchmark Function
def run_timing_trial(steps, prefill, eval_every, label):
    logdir = f'timing_bench_{label}'
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    cmd = [
        sys.executable, 'dreamer.py',
        '--configs', 'dmc_proprio',
        '--task', 'dmc_walker_walk',
        '--logdir', logdir,
        '--steps', str(steps),
        '--eval_every', str(eval_every),
        '--prefill', str(prefill),
        '--eval_episode_num', '1',
        '--seed', '0',
        '--device', device
    ]
    t0 = time.time()
    proc = subprocess.Popen(cmd)
    p = psutil.Process(proc.pid)
    peak_ram = 0.0
    peak_gpu = 0.0
    while proc.poll() is None:
        try:
            mem = p.memory_info().rss
            for c in p.children(recursive=True):
                mem += c.memory_info().rss
            peak_ram = max(peak_ram, mem / (1024 * 1024))
        except Exception:
            pass
        if torch.cuda.is_available():
            peak_gpu = max(peak_gpu, torch.cuda.max_memory_allocated() / (1024 * 1024))
        time.sleep(0.5)
    wall_sec = time.time() - t0
    sec_per_1k = (wall_sec / steps) * 1000.0
    print(f'[{label}] {steps} frames took {wall_sec:.1f}s ({sec_per_1k:.2f} s/1k frames) | Peak RAM: {peak_ram:.1f} MB | Peak GPU: {peak_gpu:.1f} MB')
    return {
        'frames': steps,
        'wall_clock_seconds': round(wall_sec, 2),
        'seconds_per_1000_frames': round(sec_per_1k, 2),
        'peak_ram_mb': round(peak_ram, 1),
        'peak_gpu_mb': round(peak_gpu, 1)
    }

In [ ]:
# 3. Execute 2,000-Frame and 5,000-Frame Timing Trials
print('Starting 2,000-frame benchmark...')
trial_2k = run_timing_trial(2000, 500, 1000, 'trial_2k')

print('Starting 5,000-frame benchmark...')
trial_5k = run_timing_trial(5000, 1000, 2000, 'trial_5k')

# Compute estimated steady-state rate from the 5k trial
sec_per_1k_est = trial_5k['seconds_per_1000_frames']
est_50k_hours = (sec_per_1k_est * 50) / 3600.0
core_plan_6runs_hours = est_50k_hours * 6.0

summary = {
    'hardware': gpu_name,
    'trial_2k': trial_2k,
    'trial_5k': trial_5k,
    'estimated_seconds_per_1000_frames': sec_per_1k_est,
    'extrapolated_50k_run_hours_ESTIMATE': round(est_50k_hours, 2),
    'extrapolated_core_plan_6runs_hours_ESTIMATE': round(core_plan_6runs_hours, 2),
    'fits_weekly_30h_quota': core_plan_6runs_hours <= 30.0
}

with open('timing_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('\n=== BENCHMARK SUMMARY ===')
print(json.dumps(summary, indent=2))
print('Saved timing_results.json')